# Task 3 — Nash Equilibrium Computation (30%)

## Objectives

By the end of this task you will:

- Implement **Counterfactual Regret Minimization (CFR)** or any other solver for single-round games.
- Measure convergence using the saddle-point residual from Task 2.

If you choose to implement CFR, you will also:

- Understand regret and regret matching in extensive-form games.
- Verify that the time-average strategy converges to an approximate Nash equilibrium.

## WARNING
Do **not** modify `treeplex_representation.py`, since we will be using it for evaluation.

## Relevant Files

| File | Role |
|------|------|
| `student_attempts/solver.py` | **Write here** — implement `CFRSolver.solve()` |
| `student_attempts/treeplex_representation.py` | Already implemented (read only) |
| `student_attempts/env_config.py` | Environment and player constants (read only) |
| `tasks/task3.ipynb` | This notebook — run cells to test |

> **Working directory**: run cells from `tasks/`. The path `..` refers to the repo root.


## Section 1 — Nash Equilibrium

A **Nash equilibrium** (NE) is a strategy profile (x\*, y\*) such that neither player can
improve their payoff by unilaterally deviating:

- Player 1: x\* is a best response to y\*
- Player 2: y\* is a best response to x\*

From Task 2, recall the **saddle-point residual**:

```
SPR(x, y) = payoff(BR₁(y), y) − payoff(x, BR₂(x))  ≥  0
```

A strategy profile (x, y) is an **ε-Nash equilibrium** if SPR(x, y) ≤ ε. The exact NE has SPR = 0. 

_Remark_: Sometimes, there is a factor of $1/2$, depending on how one defines exploitability.

### Approaches to computing Nash equilibria

| Method | Approach |
|--------|----------|
| Sequence-form LP | Solve a linear program exactly; correct but slow for large games |
| **CFR** | Iterative regret minimization; converges at O(1/√T) |
| Double Oracle | Incremental best-response generation |

- We recommend implementing **CFR**. It is iterative, scales to large games, has
strong theoretical guarantees, and we covered it in class.

- Since the game is small, sequence-form LPs may be sufficiently efficent also. We did not cover the exact
form of how the LP dual looks like, but there are many online resources to help you do so,
as far as I can tell strong LLMs work as well.

- Best responses are easy to compute for these small games, so the double oracle method may be
effective. **Warning** the ``Agent`` interface requires you to provide what are essentially behavioral
strategies, while double oracle yields a mixture of (pure) strategies. You need to mix them accordingly 
in the sequence form to recover the behavioral strategy.

## Section 2 — Walking through the CFR algorithm (ignore if you choose not to use CFR for Task 3)

For CFR, we have provided some skeletal code you may find useful inside `solvers.py`. 


### One CFR Iteration

Each iteration $t$ has three phases.

---

**Phase 1 — Regret Matching (compute current strategies)**

For each player `p`, for each info set `I`:
1. Read the cumulative regret for each action at `I`.
2. Apply regret matching to get a **behavioral distribution** `π_I` and store it.

Then convert `π_I` to sequence form and add the running sum to some accumulator. 
This allows us to eventually calculate the time-average.

The sequence form strategy calculated is what will be "played" during the self-play process. 
They corresspond to $x^{(t)}$ and $y^{(t)}$ used in our lectures.

---

**Phase 2 — Reward Vectors**

Compute reward vectors using `compute_reward_vector`:

```
rew1 = compute_reward_vector(PLAYER_1, seq_form_strategy_played[PLAYER_2])
rew2 = compute_reward_vector(PLAYER_2, seq_form_strategy_played[PLAYER_1])
```

These corresspond to $Ay$ and $A^Tx$ in our lectures.

---

**Phase 3 — Regret Update (backward traversal)**

Traverse each player's treeplex in *reversed* info set order (children before parents).
At info set `I` with action sequences `start_seq..end_seq`. The pseudocode looks something
like the following.

```
rew_obtained = Σ_a  beh_form_strategy_played[player_id][seq(I, a)]  ·  rew_p[seq(I, a)]

regrets_for_player[player_id][seq(I, a)] += rewi[seq(I, a)] − rew_obtained    for all a

rewi[parent_seq(I)] += rew_obtained    ← propagate reward upward
```

This backward pass has the same structure as the `compute_best_response` backward pass
from Task 2, but accumulates regret instead of taking argmax. Note that this backward pass 
modifies the rewards `rewi` in-place in order to take into account future rewards for 
sequences. 

Make sure you understand this step --- it is what was covered in Week 11's lecture.

---

**Convergence check**

Every once-in-a-while iterations, compute the SPR of the **time-average** strategy:

```
(sum_seq_form_pp[P1] / t,  sum_seq_form_pp[P2] / t)
```

Stop early if SPR ≤ `spr_threshold`. The time-average (not the current per-iteration
strategy) is what converges to Nash equilibrium.


## Section 3 — Task: Implement **some** solver.

If you are working with CFR, open `student_attempts/solver.py` and implement the `solve()` method.

If not, you may want to create a subclass of Solver() and work from there.

## What to submit for Task 3

We will evaluate your submission primarily based on exploitability (SPR). If your solver has converged to reasonable precision, you should score full points.

Since game solving can be a lengthy process beyond what is allowed on coursemology, we will require you to report the strategy directly. 

Tip: You will get full score for this task if the SPR is $<0.005$. 

### Converting the solver output to a playable agent

Let us assume after running your solver that you have two **sequence-form** strategy arrays. To turn them into agents that can play in the engine you need two more steps.
We will assume we want to store player 1's strategy.

1. **Sequence-form → behavioral form.**  
   The `TreeplexGame` methods expect a *behavioral* strategy (conditional probabilities at each info set, not the joint path probabilities). Use `convert_sequence_to_behavioral_form_strategy` from `treeplex_representation.py`:

   ```python
   beh_p1 = convert_sequence_to_behavioral_form_strategy(
       seq_p1.copy(), game.treeplex_p1, in_place=False)
   ```

   Pass a **copy** so the original sequence-form array is not modified.

2. **Behavioral form → `Agent` object.**  
   `TreeplexGame.behavioral_strategy_to_agent` wraps a behavioral array in an `Agent`
   that the engine can call at every decision node:

   ```python
   agent_p1 = game.behavioral_strategy_to_agent(beh_p1, PLAYER_1)
   ```

3. **Save and reload** (optional but recommended).  
   `TreeplexGame.save_agent` serialises the behavioral strategy to a `.npz` file;
   `load_agent` reconstructs the `Agent` from the same file:

   ```python
   game.save_agent(agent_p1, PLAYER_1, 'nash_p1')   # writes nash_p1.npz
   agent_p1 = game.load_agent('nash_p1.npz')        # reload later
   ```

4. **Test by playing a single round.**  
   The solver only covers one round, so use `play_round()` rather than `play_game()`.
   `play_game()` runs multiple rounds and will fail once the favour state changes:

   ```python
   from tiny_hana.tiny_hana import Game
   game_runner = Game(loaded_p1, loaded_p2, verbose=True)
   game_runner.play_round()
   print(game_runner.state.winner)  # 0, 1, or None (tie round)
   ```

The code cell below puts this all together end-to-end.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../student_attempts'))
import importlib, numpy as np
import solver as solver_module
importlib.reload(solver_module)
from solver import CFRSolver
from treeplex_representation import TreeplexGame
from treeplex_representation import convert_sequence_to_behavioral_form_strategy
from env_config import PLAYER_1, PLAYER_2
from tiny_hana.tiny_hana import Game

try:
    # ── 1. Solve ──────────────────────────────────────────────────────────
    game = TreeplexGame()
    cfr  = CFRSolver(game)
    seq_p1, seq_p2 = cfr.solve(max_iter=2000, spr_threshold=1e-3)
    print(f'Solved.  SPR = {cfr.spr:.6f}  |  game value (P1) = {cfr.val:.4f}')

    # ── 2. Sequence-form → behavioral ────────────────────────────────────
    beh_p1 = convert_sequence_to_behavioral_form_strategy(
        seq_p1.copy(), game.treeplex_p1, in_place=False)
    beh_p2 = convert_sequence_to_behavioral_form_strategy(
        seq_p2.copy(), game.treeplex_p2, in_place=False)

    # ── 3. Wrap in Agent ─────────────────────────────────────────────────
    agent_p1 = game.behavioral_strategy_to_agent(beh_p1, PLAYER_1)
    agent_p2 = game.behavioral_strategy_to_agent(beh_p2, PLAYER_2)

    # ── 4. Save to disk ──────────────────────────────────────────────────
    game.save_agent(agent_p1, PLAYER_1, 'nash_p1')
    game.save_agent(agent_p2, PLAYER_2, 'nash_p2')
    print('Saved agents to nash_p1.npz and nash_p2.npz.')

    # ── 5. Reload and verify round-trip ──────────────────────────────────
    loaded_p1 = game.load_agent('nash_p1.npz')
    loaded_p2 = game.load_agent('nash_p2.npz')
    print('Agents reloaded successfully.')

    # ── 6. Play a single round ───────────────────────────────────────────
    game_runner = Game(loaded_p1, loaded_p2, verbose=False)
    game_runner.play_round()
    result = game_runner.state.winner
    outcome = {0: 'Player 1 wins', 1: 'Player 2 wins', None: 'Tie (no winner this round)'}
    print(f'Single-round result: {outcome[result]}')

except NotImplementedError:
    print('TODO: implement CFRSolver.solve() first.')


### Generating Coursemology Submission 

You will have to submit your agents in Coursemology for evaluation.

- Run the cell above to run the solver and save your agents. 
- Run the cell below **after** saving your agents to `nash_p1.npz` and `nash_p2.npz` (by default they are saved under the same directory to this notebook). 
- Copy-paste the printed `get_model()` function into the Coursemology template.


In [ ]:
from utility import generate_get_model_snippet

AGENT1_PTH = 'nash_p1.npz'
AGENT2_PTH = 'nash_p2.npz'

print(generate_get_model_snippet(AGENT1_PTH, AGENT2_PTH))


## Tips

- Be very careful as to whether you are working with behavioral or sequence form strategies at any point!
- CFR can be optimized (code-wise) quite a bit. I suggest you stick with what I suggest, even though it is memory inefficient. You can always optimize later.
- Perform sanity checks where possible, e.g., behavioral strategies need to sum to 1, SPR should be non-negative, expected payoffs cannot be greater than 1 etc.
